## Pravila pridruživanja na skupu podataka `market.csv`

**a)** Primeniti algoritam apriori sa minimalnom podrškom od 20\% i minimalnom pouzdanošću 50\%.

**b)** Koliko pravila je ukupno pronađeno?

**c)** Koliko pravila je interesantno po lift meri?

**d)** Koliko ima pravila u kojima je u zaključku sir ('Cheese')? Izdvojiti sva takva u pravila u poseban model.

**e)** Da li je bilo potrebno podeliti podatke na trening i test skup? Da li je bilo potrebno dodatno pretprocesiranje podataka? Zašto?

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('market.csv', sep=';')
df.head()

,Bread,Honey,Bacon,Toothpaste,Banana,Apple,Hazelnut,Cheese,Meat,Carrot,...,Milk,Butter,ShavingFoam,Salt,Flour,HeavyCream,Egg,Olive,Shampoo,Sugar
0,1,0,1,0,1,1,1,0,0,1,...,0,0,0,0,0,1,1,0,0,1
1,1,1,1,0,1,1,1,0,0,0,...,1,1,0,0,1,0,0,1,1,0
2,0,1,1,1,1,1,1,1,1,0,...,1,0,1,1,1,1,1,0,0,1
3,1,1,0,1,0,1,0,0,0,0,...,1,0,0,0,1,0,1,1,1,0
4,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
transactions = [df.columns[row == 1].tolist() for _, row in df.iterrows()]
transactions[:3]

[['Bread',
  'Bacon',
  'Banana',
  'Apple',
  'Hazelnut',
  'Carrot',
  'HeavyCream',
  'Egg',
  'Sugar'],
 ['Bread',
  'Honey',
  'Bacon',
  'Banana',
  'Apple',
  'Hazelnut',
  'Cucumber',
  'Milk',
  'Butter',
  'Flour',
  'Olive',
  'Shampoo'],
 ['Honey',
  'Bacon',
  'Toothpaste',
  'Banana',
  'Apple',
  'Hazelnut',
  'Cheese',
  'Meat',
  'Cucumber',
  'Onion',
  'Milk',
  'ShavingFoam',
  'Salt',
  'Flour',
  'HeavyCream',
  'Egg',
  'Sugar']]

In [4]:
from apriori_python import apriori

In [5]:
freq_item, rules = apriori(transactions, minSup=0.2, minConf=0.5)

In [6]:
len(rules)

41

In [7]:
rules = pd.DataFrame(rules, columns=['Antecedent', 'Consequent', 'Confidence'])
rules

,Antecedent,Consequent,Confidence
0,{Honey},{Egg},0.502591
1,{HeavyCream},{Hazelnut},0.502591
2,{Egg},{Carrot},0.502674
3,{Egg},{Bacon},0.502674
4,{Salt},{Cheese},0.502703
5,{Cheese},{Bacon},0.504854
6,{Cheese},{Banana},0.504854
7,{Bread},{Salt},0.507937
8,{Honey},{Banana},0.512953
9,{Bacon},{Hazelnut},0.515000


In [8]:
def support(itemset, transactions):
    count = sum(itemset.issubset(t) for t in transactions)
    return count / len(transactions)

In [9]:
transactions = [set(t) for t in transactions]

In [10]:
rules['Lift'] = rules.apply(lambda row : row['Confidence'] / support(row['Consequent'], transactions), axis=1)

In [11]:
rules

,Antecedent,Consequent,Confidence,Lift
0,{Honey},{Egg},0.502591,1.247070
1,{HeavyCream},{Hazelnut},0.502591,1.195908
2,{Egg},{Carrot},0.502674,1.214795
3,{Egg},{Bacon},0.502674,1.166203
4,{Salt},{Cheese},0.502703,1.132301
5,{Cheese},{Bacon},0.504854,1.171262
6,{Cheese},{Banana},0.504854,1.126214
7,{Bread},{Salt},0.507937,1.273960
8,{Honey},{Banana},0.512953,1.144281
9,{Bacon},{Hazelnut},0.515000,1.225436


In [15]:
rules['Lift'].min()

1.1262135922330097

Sva pravila su interesantna po lift meri jer je minimalni lift 1.13.

In [13]:
cheese_rules = rules[rules.apply(lambda row : {'Cheese'}.issubset(row['Consequent']), axis=1)]
cheese_rules

,Antecedent,Consequent,Confidence,Lift
4,{Salt},{Cheese},0.502703,1.132301
15,{Bacon},{Cheese},0.520000,1.171262
17,{Honey},{Cheese},0.523316,1.178731
18,{Carrot},{Cheese},0.526042,1.184871
19,{ShavingFoam},{Cheese},0.526596,1.186119
20,{Meat},{Cheese},0.527778,1.188781
25,{Bread},{Cheese},0.529101,1.191760
34,{Egg},{Cheese},0.550802,1.240642
40,{Butter},{Cheese},0.574713,1.294498


In [14]:
len(cheese_rules)

9

Nije potrebno deliti podatke na trening i test skup, s obzirom na to da ne primenjujemo algoritam nadgledanog učenja. Pretprocesiranje je bilo potrebno samo kako bi se podaci iz učitanog formata pripremili u vidu liste transakcija koje očekuje apriori algoritam.